### Load dependencies

In [1]:
import os
import json
import sympy
import torch
import numpy as np
import pandas as pd
from typing import Dict, List, Callable

# MIRA modeling dependencies
from mira.metamodel import *
from mira.metamodel.ops import stratify
from mira.examples.concepts import susceptible, exposed, infected, recovered
from mira.modeling import Model
from mira.modeling.amr.petrinet import AMRPetriNetModel, template_model_to_petrinet_json
from mira.sources.amr.petrinet import template_model_from_amr_json
from mira.metamodel.io import model_to_json_file, model_from_json_file

# PyCIEMSS dependencies
import pyciemss
import pyciemss.visuals.plots as plots
import pyciemss.visuals.vega as vega
import pyciemss.visuals.trajectories as trajectories
from pyciemss.integration_utils.intervention_builder import (
    param_value_objective,
    start_time_objective,
)

### Select a model

In [9]:
person_units = lambda: Unit(expression=sympy.Symbol('person'))
day_units = lambda: Unit(expression=sympy.Symbol('day'))
per_day_units = lambda: Unit(expression=1/sympy.Symbol('day'))
dimensionless_units = lambda: Unit(expression=sympy.Integer('1'))
per_day_per_person_units = lambda: Unit(expression=1/(sympy.Symbol('day')*sympy.Symbol('person')))

c = {
    'S': Concept(name='S', units=person_units(), identifiers={'ido': '0000514'}),
    'I': Concept(name='I', units=person_units(), identifiers={'ido': '0000511'}),
    'R': Concept(name='R', units=person_units(), identifiers={'ido': '0000592'}),
    'I_deriv': Concept(name='I_deriv', units=person_units(), identifiers={'ido': '0000502'})
}

for concept in c:
    c[concept].name = concept

parameters = {
    'beta': Parameter(name='beta', value=sympy.Float(0.35), units=per_day_units(),
                      distribution=Distribution(type='StandardUniform1',
                                                parameters={'minimum': 0.25,
                                                            'maximum': 0.45})),  # Transmission rate
    'N': Parameter(name='total_population', value=sympy.Float(1001), units=person_units()),  # Total population

    'p1': Parameter(name='p1', value=sympy.Float(1.0), units=per_day_units(),
                     distribution=Distribution(type='StandardUniform1',
                                                parameters={'minimum': 0.9999,
                                                            'maximum': 1.0001})),  # Multiplier
    
    'tr': Parameter(name='tr', value=sympy.Float(14), units=per_day_units(),
                     distribution=Distribution(type='StandardUniform1',
                                                parameters={'minimum': 8.0,
                                                            'maximum': 20.0})),  # Infectious period
}

S, I, R, I_deriv, beta, N, p1, tr = sympy.symbols('S I R I_deriv beta N p1 tr')

initials = {
    "S": Initial(concept=c["S"], expression=1000),
    "I": Initial(concept=c["I"], expression=1),
    "R": Initial(concept=c["R"], expression=0),
    "I_deriv": Initial(concept=c["I_deriv"], expression=0),
}

##### S -> I
si = ControlledConversion(
    subject=c['S'],
    outcome=c['I'],
    controller=c['I'],
    rate_law=beta*p1*S*I / N
)


#### I -> R
ir = NaturalConversion(
    subject=c['I'],
    outcome=c['R'],
    rate_law=(1 / tr)*I
)


#### I_deriv
id = GroupedControlledProduction(
    controllers=[c['I'], c['S']],
    outcome=c['I_deriv'],
    rate_law=beta**2*p1**2*S**2*I/N**2 - beta*p1*S*I**2/N**2 - 2*beta*p1*S*I*tr/N + 1/tr**2
)

sir_model = TemplateModel(
    templates=[
        si,
        ir,
        id
    ],
    parameters=parameters,
    initials=initials,
    time=Time(name='t', units=day_units()),
    observables=[],
    annotations=Annotations(name='SIR')
)

# Save as JSON
with open("SIR_petrinet.json", 'w') as fh:
    json.dump(template_model_to_petrinet_json(sir_model), fh, indent=1)

## Simulate the model with no interventions

In [10]:
model1 = "SIR_petrinet.json"

start_time = 0.0
end_time = 40.0
logging_step_size = 1.0
num_samples = 100

result1 = pyciemss.sample(model1, end_time, logging_step_size, num_samples, start_time=start_time)
display(result1['data'].head())

# Plot results for all states
schema = plots.trajectories(result1["data"], keep=".*_state")
plots.save_schema(schema, "_schema.json")
plots.ipy_display(schema, dpi=150)

ERROR:root:
                ###############################

                There was an exception in pyciemss

                Error occured in function: sample

                Function docs : 
    Load a model from a file, compile it into a probabilistic program, and sample from it.

    Args:
        model_path_or_json: Union[str, Dict]
            - A path to a AMR model file or JSON containing a model in AMR form.
        end_time: float
            - The end time of the sampled simulation.
        logging_step_size: float
            - The step size to use for logging the trajectory.
        num_samples: int
            - The number of samples to draw from the model.
        solver_method: str
            - The method to use for solving the ODE. See torchdiffeq's `odeint` method for more details.
            - If performance is incredibly slow, we suggest using `euler` to debug.
              If using `euler` results in faster simulation, the issue is likely that the model is s

TypeError: `y0` must be a floating point Tensor but is a torch.LongTensor

In [ ]:
num_samples = 1
end_time = 100.0

# Define the threshold for when the intervention should be applied
def make_var_threshold(var: str, threshold: torch.Tensor):
    previous_value = torch.tensor(1.1)

    def threshold_function(time, state):
        nonlocal previous_value
        if previous_value is torch.tensor(1.1):
            previous_value = state[var]
            return torch.tensor(1.0)  # On the first call, there's no previous value to compare to

        if previous_value < threshold <= state[var]:
            result = torch.tensor(0.0)  # Crosses from below
        elif previous_value > threshold >= state[var]:
            result = torch.tensor(1.0)  # Crosses from above
        else:
            result = torch.tensor(1.0)  # No crossing

        previous_value = state[var]
        return result

    return threshold_function


# Define the threshold for when the intervention should be applied
def make_var_threshold2(var: str, threshold: torch.Tensor):
    previous_value2 = torch.tensor(1.1)

    def threshold_function2(time, state):
        nonlocal previous_value2
        if previous_value2 is torch.tensor(1.1):
            previous_value2 = state[var]
            return torch.tensor(1.0)  # On the first call, there's no previous value to compare to

        if previous_value2 < threshold <= state[var]:
            result = torch.tensor(1.0)  # Crosses from below
        elif previous_value2 > threshold >= state[var]:
            result = torch.tensor(0.0)  # Crosses from above
        else:
            result = torch.tensor(1.0)  # No crossing

        previous_value2 = state[var]
        return result

    return threshold_function2

infection_threshold = make_var_threshold("I", torch.tensor(150.0))
infection_threshold2 = make_var_threshold2("I", torch.tensor(155.0))
# dynamic_parameter_interventions1 = {infection_threshold: {"p_cbeta": torch.tensor(0.3)},
#                                     infection_threshold2: {"p_cbeta": torch.tensor(0.45)}}

dynamic_parameter_interventions1 = {infection_threshold2: {"p_cbeta": torch.tensor(0.3)}}

result = pyciemss.sample(model1, end_time, logging_step_size, num_samples, start_time=start_time, 
                         dynamic_parameter_interventions=dynamic_parameter_interventions1, 
                         solver_method="dopri5")
display(result["data"].head())

# Plot the result
schema = plots.trajectories(pd.DataFrame(result["data"]), keep=".*_state")
plots.save_schema(schema, "_schema.json")
plots.ipy_display(schema, dpi=150)

In [ ]:
schema = plots.trajectories(pd.DataFrame(result["data"]), keep="persistent_p_cbeta_param")
plots.save_schema(schema, "_schema.json")
plots.ipy_display(schema, dpi=150)